# Figure: accuracy and precision

The cell below is tagged `#| label: fig-precision-output`, so `04-issues.md` embeds its output with `:::{figure} #fig-precision-output`. The simulation is in `cardiac_sim.py` in this folder.

In [ ]:
#| label: fig-precision-output
# Accuracy vs precision: each method is simulated many times with random noise.
# The simulation itself is in cardiac_sim.py (shared with the other figures).
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings, logging
# Kaleido (the PNG export for the PDF) can print harmless clean-up warnings on Windows.
warnings.filterwarnings("ignore", message=".*temporary directory.*")
for name in ("choreographer", "kaleido", "logistro"):
    logging.getLogger(name).setLevel(logging.CRITICAL)
import os, sys
sys.path.append(os.path.join(os.getcwd(), "notebooks"))  # find cardiac_sim.py also when run from the book folder
from cardiac_sim import SEQUENCES, simulate, fit

# Interactive figure for the website + PNG snapshot for the PDF (same probe as the template).
try:
    pio.to_image(go.Figure(), format="png")
    pio.renderers.default = "plotly_mimetype+png"
except Exception as exc:
    pio.renderers.default = "plotly_mimetype"
    print(f"Static PNG export unavailable ({type(exc).__name__}); PDF will show the caption only.")

T1 = 1000.0                # native myocardium [ms]
RR = 1000.0                # 60 beats per minute
SNR_VALUES = list(range(10, 105, 5))  # SNR from 10 to 100 in steps of 5 (M0 / noise SD)
N_REPEATS = 500            # number of simulated scans per method and SNR
COLORS = {"MOLLI": "royalblue", "ShMOLLI": "seagreen", "SASHA": "darkorange"}

# one panel per method, side by side, with the same axes
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=SEQUENCES)
titles = list(fig.layout.annotations)          # keep the panel titles when the slider changes
notes_per_snr = []
for j, snr in enumerate(SNR_VALUES):
    notes = []
    rng = np.random.default_rng(seed=snr)   # fixed seed per SNR, so the figure is always the same
    for col, seq in enumerate(SEQUENCES, start=1):
        _, _, _, samples = simulate(seq, T1, RR)
        estimates = np.array([fit(seq, samples, RR, rng.normal(0, 1 / snr, len(samples)))[2]
                              for _ in range(N_REPEATS)])
        fig.add_trace(go.Histogram(x=estimates, xbins=dict(start=500, end=1500, size=10),
                                   marker_color=COLORS[seq], showlegend=False,
                                   visible=(snr == 40)), row=1, col=col)
        # mean ± SD written into the panel
        axis = "" if col == 1 else str(col)
        notes.append(dict(xref=f"x{axis} domain", yref=f"y{axis} domain", x=0.03, y=0.97,
                          xanchor="left", yanchor="top", showarrow=False, align="left",
                          text=f"{estimates.mean():.0f} ± {estimates.std():.0f} ms",
                          bgcolor="white", font=dict(size=13)))
    notes_per_snr.append(notes)

for col in range(1, 4):                         # true T1 in every panel
    fig.add_vline(x=T1, line=dict(color="black", dash="dash", width=1), row=1, col=col)

steps = []
for j, snr in enumerate(SNR_VALUES):
    visible = [False] * (len(SNR_VALUES) * len(SEQUENCES))
    visible[j * len(SEQUENCES):(j + 1) * len(SEQUENCES)] = [True] * len(SEQUENCES)
    steps.append(dict(method="update", label=str(snr),
                      args=[{"visible": visible}, {"annotations": titles + notes_per_snr[j]}]))

fig.update_layout(
    height=420, template="plotly_white", margin=dict(t=50, b=120),
    annotations=titles + notes_per_snr[SNR_VALUES.index(40)],
    sliders=[dict(active=SNR_VALUES.index(40), steps=steps,
                  currentvalue=dict(prefix="SNR: "), pad=dict(t=50))],
)
fig.update_xaxes(range=[600, 1400], title_text="Fitted T1 [ms]")
fig.update_yaxes(title_text="Number of simulated scans", row=1, col=1)

# Only show values that matter: no coordinates on hover, no zooming.
fig.update_layout(hovermode=False, dragmode=False)
fig.update_xaxes(fixedrange=True)
fig.update_yaxes(fixedrange=True)
fig
